<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/FHIRy_pyOMOP_TFL_Colab_Clean_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FHIRy–pyOMOP Transformation Fidelity

FHIR R4 → FHIRy → pandas → pyOMOP → OMOP CDM

This notebook adds a sidecar transformation-fidelity audit with source-to-target lineage,
information-fate categories, deterministic warnings, and summary metrics.

# Phase A

## Environment setup

In [1]:
%pip -q install pyomop==6.4.0 fhiry==5.2.2 pyarrow scipy matplotlib tabulate nest-asyncio

import sys, os, json, math, time, re, ast, hashlib, uuid, shutil, sqlite3, platform, tempfile, asyncio
from pathlib import Path
from collections import defaultdict
from enum import Enum

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon

import pyomop
import fhiry

print("Python:", sys.version.split()[0])
print("pyomop:", getattr(pyomop, "__version__", "unknown"))
print("fhiry:", getattr(fhiry, "__version__", "unknown"))
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 5.7 MB/s eta 0:00:00


Python: 3.12.13
pyomop: 6.4.0
fhiry: 5.2.2
pandas: 2.2.2
numpy: 2.0.2


## Google Drive and repository

In [2]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git"
REPO_DIR = Path("/content/ohdsi-fhir-omop-showcase-demo")

MYDRIVE = Path("/content/drive/MyDrive")
PROJECT_DRIVE_DIR = MYDRIVE / "fhir_omop_colab"

if not REPO_DIR.exists():
    os.system(f'git clone -q "{REPO_URL}" "{REPO_DIR}"')
else:
    print("Repository already exists:", REPO_DIR)

PROJECT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

TFL_DRIVE_DIR = PROJECT_DRIVE_DIR / "tfl_runs"
TFL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

PUBLIC_OUT = REPO_DIR / "results" / "transformation_fidelity"
PUBLIC_OUT.mkdir(parents=True, exist_ok=True)

FIG_DIR = REPO_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_DB_NAMES = {
    "V0": "V0_clinical_core_25k.sqlite",
    "V1": "V1_missing_demographics_clinical_core_25k.sqlite",
    "V2": "V2_duplicate_encounter_ids_clinical_core_25k.sqlite",
    "V3": "V3_conflicting_codings_clinical_core_25k.sqlite",
    "V4": "V4_missing_medications_clinical_core_25k.sqlite",
    "V5": "V5_missing_demographics_plus_conflicting_codings_clinical_core_25k.sqlite",
}

def find_database_files(root):
    files = []
    for pattern in ("*.sqlite", "*.sqlite3", "*.db"):
        for p in root.rglob(pattern):
            try:
                if p.is_file() and p.stat().st_size > 0:
                    text = str(p).lower()
                    if "/tfl_runs/" not in text and "tfl_fresh" not in text:
                        files.append(p)
            except OSError:
                pass
    return sorted(set(files), key=lambda p: str(p))

ALL_DATABASE_FILES = find_database_files(MYDRIVE)

print(f"Non-empty database files found: {len(ALL_DATABASE_FILES)}")

def choose_variant_database(variant):
    expected_name = EXPECTED_DB_NAMES[variant].lower()

    # 1. Exact filename anywhere in MyDrive.
    exact = [
        p for p in ALL_DATABASE_FILES
        if p.name.lower() == expected_name
    ]
    if len(exact) == 1:
        return exact[0], "exact_filename"
    if len(exact) > 1:
        exact = sorted(exact, key=lambda p: p.stat().st_mtime, reverse=True)
        return exact[0], "exact_filename_multiple_newest"

    # 2. Variant-specific fallback.
    variant_token = variant.lower()
    candidates = []

    for p in ALL_DATABASE_FILES:
        text = str(p).lower()
        name = p.name.lower()

        if not re.search(rf'(^|[_\-/]){variant_token}([_\-.]|$)', text):
            continue

        score = 0

        if "clinical_core_25k" in name:
            score += 10

        if variant == "V1" and "missing_demograph" in text:
            score += 10

        if variant == "V2" and "duplicate_encounter" in text:
            score += 10

        if variant == "V3" and "conflicting_coding" in text:
            score += 10

        if variant == "V4" and (
            "missing_medication" in text
            or "medication" in text
        ):
            score += 10

        if variant == "V5" and (
            "missing_demograph" in text
            and "conflicting_coding" in text
        ):
            score += 15

        candidates.append((score, p))

    if not candidates:
        return None, "not_found"

    candidates.sort(
        key=lambda x: (x[0], x[1].stat().st_mtime),
        reverse=True
    )

    best_score = candidates[0][0]
    best = [p for score, p in candidates if score == best_score]

    if len(best) == 1:
        return best[0], "variant_match"

    best.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return best[0], "variant_match_multiple_newest"

VALIDATED_DB = {}
DB_RESOLUTION_ROWS = []

for variant in ["V0", "V1", "V2", "V3", "V4", "V5"]:
    path, method = choose_variant_database(variant)

    if path is not None:
        VALIDATED_DB[variant] = path

    DB_RESOLUTION_ROWS.append({
        "variant": variant,
        "resolution": method,
        "path": str(path) if path else None,
        "size_mb": (
            round(path.stat().st_size / (1024**2), 3)
            if path else None
        ),
    })

db_resolution = pd.DataFrame(DB_RESOLUTION_ROWS)
display(db_resolution)

Mounted at /content/drive
Non-empty database files found: 2


,variant,resolution,path,size_mb
0,V0,exact_filename,/content/drive/MyDrive/fhir_omop_colab/results...,27.531
1,V1,variant_match,/content/drive/MyDrive/fhir_omop_colab/results...,27.531
2,V2,exact_filename,/content/drive/MyDrive/fhir_omop_colab/results...,27.531
3,V3,not_found,None,NaN
4,V4,variant_match,/content/drive/MyDrive/fhir_omop_colab/results...,27.531
5,V5,not_found,None,NaN


In [3]:
# Use this cell only when a database exists but has a different filename/path.
# Enter only the variants that were not resolved automatically.

MANUAL_DB_OVERRIDES = {
    # "V1": Path("/content/drive/MyDrive/.../your_V1_file.sqlite"),
    # "V3": Path("/content/drive/MyDrive/.../your_V3_file.sqlite"),
    # "V4": Path("/content/drive/MyDrive/.../your_V4_file.sqlite"),
    # "V5": Path("/content/drive/MyDrive/.../your_V5_file.sqlite"),
}

for variant, path in MANUAL_DB_OVERRIDES.items():
    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"{variant}: invalid database path: {path}")

    VALIDATED_DB[variant] = path

print("Current database map:")
for variant in ["V0", "V1", "V2", "V3", "V4", "V5"]:
    print(variant, "->", VALIDATED_DB.get(variant))

Current database map:
V0 -> /content/drive/MyDrive/fhir_omop_colab/results_25k/V0_clinical_core_25k.sqlite
V1 -> /content/drive/MyDrive/fhir_omop_colab/results_V1_V4_25k/V2_duplicate_encounter_ids_clinical_core_25k.sqlite
V2 -> /content/drive/MyDrive/fhir_omop_colab/results_V1_V4_25k/V2_duplicate_encounter_ids_clinical_core_25k.sqlite
V3 -> None
V4 -> /content/drive/MyDrive/fhir_omop_colab/results_V1_V4_25k/V2_duplicate_encounter_ids_clinical_core_25k.sqlite
V5 -> None


Phase A uses V0 only. V1–V5 are required in the later multi-variant phases.

## Locate V0–V5 databases

In [4]:
def verify_sqlite_files(db_map):
    rows = []

    for variant in ["V0", "V1", "V2", "V3", "V4", "V5"]:
        path = db_map.get(variant)

        exists = bool(path and Path(path).exists())
        size = Path(path).stat().st_size if exists else 0
        valid = exists and size > 0

        rows.append({
            "variant": variant,
            "path": str(path) if path else None,
            "exists": exists,
            "bytes": size,
            "valid_nonempty": valid,
        })

    report = pd.DataFrame(rows)
    display(report)

    return report

db_file_report = verify_sqlite_files(VALIDATED_DB)

V0_READY = bool(
    len(db_file_report) >= 1
    and db_file_report.loc[
        db_file_report["variant"] == "V0",
        "valid_nonempty"
    ].any()
)

ALL_VARIANTS_READY = bool(
    len(db_file_report) == 6
    and db_file_report["valid_nonempty"].all()
)

print("V0_READY =", V0_READY)
print("ALL_VARIANTS_READY =", ALL_VARIANTS_READY)

if not V0_READY:
    print(
        "V0 is not resolved. Check db_resolution above and set "
        "VALIDATED_DB['V0'] manually if the file has a different name."
    )

if V0_READY and not ALL_VARIANTS_READY:
    unresolved = db_file_report.loc[
        ~db_file_report["valid_nonempty"],
        "variant"
    ].tolist()

    print(
        "Phase A can continue with V0. "
        "Unresolved variants for later phases: "
        + ", ".join(unresolved)
    )

,variant,path,exists,bytes,valid_nonempty
0,V0,/content/drive/MyDrive/fhir_omop_colab/results...,True,28868608,True
1,V1,/content/drive/MyDrive/fhir_omop_colab/results...,True,28868608,True
2,V2,/content/drive/MyDrive/fhir_omop_colab/results...,True,28868608,True
3,V3,None,False,0,False
4,V4,/content/drive/MyDrive/fhir_omop_colab/results...,True,28868608,True
5,V5,None,False,0,False


V0_READY = True
ALL_VARIANTS_READY = False
Phase A can continue with V0. Unresolved variants for later phases: V3, V5


## V0 baseline checks

In [5]:
if not V0_READY:
    raise FileNotFoundError(
        "V0 database is not resolved. Review db_resolution and set "
        "VALIDATED_DB['V0'] to the correct non-empty SQLite file."
    )

CORE_TABLES = [
    "person",
    "visit_occurrence",
    "condition_occurrence",
    "drug_exposure",
    "measurement",
    "observation",
    "procedure_occurrence",
]

EXPECTED_V0_ROWS = {
    "person": 1071,
    "visit_occurrence": 25000,
    "condition_occurrence": 25000,
    "drug_exposure": 25000,
    "measurement": 25000,
    "observation": 25000,
    "procedure_occurrence": 25000,
}

def table_exists(conn, table):
    q = "SELECT 1 FROM sqlite_master WHERE type='table' AND lower(name)=lower(?) LIMIT 1"
    return conn.execute(q, (table,)).fetchone() is not None

def table_columns(conn, table):
    if not table_exists(conn, table):
        return []
    return [r[1] for r in conn.execute(f'PRAGMA table_info("{table}")').fetchall()]

def primary_key_columns(conn, table):
    if not table_exists(conn, table):
        return []
    rows = conn.execute(f'PRAGMA table_info("{table}")').fetchall()
    return [r[1] for r in rows if int(r[5] or 0) > 0]

def scalar(conn, sql, params=()):
    return conn.execute(sql, params).fetchone()[0]

def sha256_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

v0_path = VALIDATED_DB["V0"]

with sqlite3.connect(v0_path) as conn:
    baseline_rows = []

    for table in CORE_TABLES:
        n = (
            scalar(conn, f'SELECT COUNT(*) FROM "{table}"')
            if table_exists(conn, table)
            else None
        )

        expected = EXPECTED_V0_ROWS.get(table)

        baseline_rows.append({
            "table": table,
            "rows": n,
            "expected_rows": expected,
            "matches_expected": (
                n == expected
                if n is not None
                else False
            ),
            "pk_columns": "|".join(
                primary_key_columns(conn, table)
            ),
        })

v0_table_counts = pd.DataFrame(baseline_rows)
display(v0_table_counts)

,table,rows,expected_rows,matches_expected,pk_columns
0,person,1071,1071,True,person_id
1,visit_occurrence,25000,25000,True,visit_occurrence_id
2,condition_occurrence,25000,25000,True,condition_occurrence_id
3,drug_exposure,25000,25000,True,drug_exposure_id
4,measurement,25000,25000,True,measurement_id
5,observation,25000,25000,True,observation_id
6,procedure_occurrence,25000,25000,True,procedure_occurrence_id


In [6]:
with sqlite3.connect(v0_path) as conn:
    pcols = set(table_columns(conn, "person"))

    v0_gender = pd.read_sql_query(
        """
        SELECT
            COALESCE(CAST(gender_concept_id AS TEXT), '__NULL__') AS gender_concept_id,
            COALESCE(NULLIF(TRIM(CAST(gender_source_value AS TEXT)), ''), '__MISSING__') AS gender_source_value,
            COUNT(*) AS n
        FROM person
        GROUP BY 1,2
        ORDER BY n DESC
        """,
        conn,
    )

    demo = {
        "persons": scalar(conn, "SELECT COUNT(*) FROM person"),
        "missing_or_unknown_gender_concept": (
            scalar(conn, "SELECT COUNT(*) FROM person WHERE gender_concept_id IS NULL OR gender_concept_id=0")
            if "gender_concept_id" in pcols else np.nan
        ),
        "missing_gender_source": (
            scalar(conn, "SELECT COUNT(*) FROM person WHERE gender_source_value IS NULL OR TRIM(CAST(gender_source_value AS TEXT))=''")
            if "gender_source_value" in pcols else np.nan
        ),
        "missing_or_zero_birth_year": (
            scalar(conn, "SELECT COUNT(*) FROM person WHERE year_of_birth IS NULL OR year_of_birth=0")
            if "year_of_birth" in pcols else np.nan
        ),
    }

v0_demographic_sanity = pd.DataFrame([demo])
display(v0_demographic_sanity)
display(v0_gender.head(20))

,persons,missing_or_unknown_gender_concept,missing_gender_source,missing_or_zero_birth_year
0,1071,0,0,0


,gender_concept_id,gender_source_value,n
0,8532,female,551
1,8507,male,520


In [7]:
DOMAIN_SPECS = {
    "condition_occurrence": ("condition_concept_id", "condition_source_value"),
    "drug_exposure": ("drug_concept_id", "drug_source_value"),
    "measurement": ("measurement_concept_id", "measurement_source_value"),
    "observation": ("observation_concept_id", "observation_source_value"),
    "visit_occurrence": ("visit_concept_id", "visit_source_value"),
    "procedure_occurrence": ("procedure_concept_id", "procedure_source_value"),
}

mapping_rows = []
with sqlite3.connect(v0_path) as conn:
    for table, (concept_col, source_col) in DOMAIN_SPECS.items():
        cols = set(table_columns(conn, table))
        if not {concept_col, source_col}.issubset(cols):
            continue

        total = scalar(conn, f'SELECT COUNT(*) FROM "{table}"')
        mapped = scalar(
            conn,
            f'SELECT COUNT(*) FROM "{table}" WHERE "{concept_col}" IS NOT NULL AND "{concept_col}" <> 0'
        )
        unique_source = scalar(
            conn,
            f"""SELECT COUNT(DISTINCT COALESCE(NULLIF(TRIM(CAST("{source_col}" AS TEXT)),''),'__MISSING__'))
                FROM "{table}" """
        )
        missing_source = scalar(
            conn,
            f"""SELECT COUNT(*) FROM "{table}"
                WHERE "{source_col}" IS NULL OR TRIM(CAST("{source_col}" AS TEXT))='' """
        )

        mapping_rows.append({
            "table": table,
            "total_rows": total,
            "mapped_concept_rows": mapped,
            "mapped_percent": 100.0 * mapped / total if total else np.nan,
            "unique_source_values_including_missing": unique_source,
            "missing_source_value_rows": missing_source,
        })

v0_mapping_sanity = pd.DataFrame(mapping_rows)
display(v0_mapping_sanity)

,table,total_rows,mapped_concept_rows,mapped_percent,unique_source_values_including_missing,missing_source_value_rows
0,condition_occurrence,25000,0,0.0,247,0
1,drug_exposure,25000,0,0.0,147,9687
2,measurement,25000,0,0.0,180,0
3,observation,25000,0,0.0,180,0
4,visit_occurrence,25000,0,0.0,5,0
5,procedure_occurrence,25000,0,0.0,274,0


In [8]:
# Optional state-of-residence check if person→location and a state field are present.
loc_summary = None

with sqlite3.connect(v0_path) as conn:
    person_cols = set(table_columns(conn, "person"))
    location_cols = set(table_columns(conn, "location")) if table_exists(conn, "location") else set()

    if "location_id" in person_cols and "location_id" in location_cols:
        state_candidates = [c for c in ["state", "state_source_value"] if c in location_cols]
        if state_candidates:
            state_col = state_candidates[0]
            loc_summary = pd.read_sql_query(
                f"""
                SELECT
                    COALESCE(NULLIF(TRIM(CAST(l."{state_col}" AS TEXT)),''),'__MISSING__') AS state,
                    COUNT(DISTINCT p.person_id) AS persons
                FROM person p
                LEFT JOIN location l ON p.location_id=l.location_id
                GROUP BY 1
                ORDER BY persons DESC
                """,
                conn,
            )

if loc_summary is None:
    print("State summary skipped: necessary location/state fields are not exposed in this V0 schema.")
else:
    display(loc_summary.head(30))

,state,persons
0,__MISSING__,1071


In [9]:
pk_rows = []
with sqlite3.connect(v0_path) as conn:
    for table in CORE_TABLES:
        if not table_exists(conn, table):
            continue
        pks = primary_key_columns(conn, table)
        if len(pks) == 1:
            pk = pks[0]
            total = scalar(conn, f'SELECT COUNT(*) FROM "{table}"')
            distinct_pk = scalar(conn, f'SELECT COUNT(DISTINCT "{pk}") FROM "{table}"')
            pk_rows.append({
                "table": table,
                "primary_key": pk,
                "rows": total,
                "distinct_primary_keys": distinct_pk,
                "unique": total == distinct_pk,
            })

v0_pk_sanity = pd.DataFrame(pk_rows)

v0_manifest = pd.DataFrame([{
    "baseline": "V0",
    "database_path": str(v0_path),
    "database_bytes": v0_path.stat().st_size,
    "sha256": sha256_file(v0_path),
    "person_n": int(v0_table_counts.loc[v0_table_counts["table"]=="person", "rows"].iloc[0]),
    "transformation_role": "sole synthetic baseline",
}])

display(v0_pk_sanity)
display(v0_manifest)

,table,primary_key,rows,distinct_primary_keys,unique
0,person,person_id,1071,1071,True
1,visit_occurrence,visit_occurrence_id,25000,25000,True
2,condition_occurrence,condition_occurrence_id,25000,25000,True
3,drug_exposure,drug_exposure_id,25000,25000,True
4,measurement,measurement_id,25000,25000,True
5,observation,observation_id,25000,25000,True
6,procedure_occurrence,procedure_occurrence_id,25000,25000,True


,baseline,database_path,database_bytes,sha256,person_n,transformation_role
0,V0,/content/drive/MyDrive/fhir_omop_colab/results...,28868608,1c00b403f168c63088237247fc3cc7300730620003d07b...,1071,sole synthetic baseline


In [10]:
# Export Phase A outputs.
v0_table_counts.to_csv(PUBLIC_OUT / "phaseA_v0_table_counts.csv", index=False)
v0_demographic_sanity.to_csv(PUBLIC_OUT / "phaseA_v0_demographic_sanity.csv", index=False)
v0_gender.to_csv(PUBLIC_OUT / "phaseA_v0_gender_counts.csv", index=False)
v0_mapping_sanity.to_csv(PUBLIC_OUT / "phaseA_v0_mapping_sanity.csv", index=False)
v0_pk_sanity.to_csv(PUBLIC_OUT / "phaseA_v0_primary_key_sanity.csv", index=False)
v0_manifest.to_csv(PUBLIC_OUT / "phaseA_v0_manifest.csv", index=False)

if loc_summary is not None:
    loc_summary.to_csv(PUBLIC_OUT / "phaseA_v0_state_counts.csv", index=False)

print("Phase A outputs:", PUBLIC_OUT)

Phase A outputs: /content/ohdsi-fhir-omop-showcase-demo/results/transformation_fidelity


# Phase B

## Locate V0–V5 FHIR source inputs

In [11]:
print("Searching FHIR inputs under:", MYDRIVE)
if not MYDRIVE.exists():
    raise FileNotFoundError(f"Google Drive root not found: {MYDRIVE}")

VARIANT_TOKENS = {
    "V0": ["v0", "baseline", "clean"],
    "V1": ["v1", "missing", "demographic"],
    "V2": ["v2", "duplicate", "encounter"],
    "V3": ["v3", "conflicting", "coding"],
    "V4": ["v4", "medication"],
    "V5": ["v5", "combined", "missing", "coding"],
}

def source_candidates(root):
    root = Path(root)
    candidates = set()
    if not root.exists():
        return []

    for f in root.rglob("*.ndjson"):
        low = str(f).lower()
        if "mimic" not in low:
            candidates.add(f.parent)

    for ext in ("*.csv", "*.parquet"):
        for f in root.rglob(ext):
            low = str(f).lower()
            if any(x in low for x in ["mimic", "results_reviewer", "analytical_stability", "transformation_fidelity"]):
                continue
            if any(x in f.name.lower() for x in ["fhir", "flatten", "variant", "v0", "v1", "v2", "v3", "v4", "v5"]):
                candidates.add(f)

    return sorted(candidates, key=lambda p: str(p))

def score_candidate(variant, path):
    s = str(path).lower()
    score = 0
    if variant.lower() in s:
        score += 8
    for tok in VARIANT_TOKENS[variant][1:]:
        if tok in s:
            score += 2
    return score

candidate_rows = []
for p in source_candidates(MYDRIVE):
    for variant in VARIANT_TOKENS:
        score = score_candidate(variant, p)
        if score > 0:
            candidate_rows.append({
                "variant": variant,
                "score": score,
                "candidate": str(p),
            })

candidate_df = pd.DataFrame(candidate_rows)
if len(candidate_df):
    candidate_df = candidate_df.sort_values(["variant","score"], ascending=[True,False])

display(candidate_df.head(100) if len(candidate_df) else candidate_df)

Searching FHIR inputs under: /content/drive/MyDrive


,variant,score,candidate
226,V0,8,/content/drive/MyDrive/MCI_Project/mci-cardiom...
227,V0,8,/content/drive/MyDrive/MCI_Project/mci-cardiom...
228,V0,8,/content/drive/MyDrive/MCI_Project/mci-cardiom...
229,V0,8,/content/drive/MyDrive/MCI_Project/mci-cardiom...
230,V0,8,/content/drive/MyDrive/MCI_Project/mci-cardiom...
...,...,...,...
73,V1,8,/content/drive/MyDrive/GES_RAG_Temporal_Study/...
74,V1,8,/content/drive/MyDrive/GES_RAG_Temporal_Study/...
75,V1,8,/content/drive/MyDrive/GES_RAG_Temporal_Study/...
76,V1,8,/content/drive/MyDrive/GES_RAG_Temporal_Study/...


In [12]:
# Set source paths when they are identified.
VARIANT_SOURCE_INPUTS = {v: None for v in ["V0","V1","V2","V3","V4","V5"]}

# Conservative auto-fill: exactly one top candidate with strong score.
if len(candidate_df):
    for v in VARIANT_SOURCE_INPUTS:
        sub = candidate_df[candidate_df["variant"] == v]
        if len(sub):
            top_score = sub["score"].max()
            top = sub[sub["score"] == top_score]
            if top_score >= 10 and len(top) == 1:
                VARIANT_SOURCE_INPUTS[v] = Path(top.iloc[0]["candidate"])

for v, p in VARIANT_SOURCE_INPUTS.items():
    print(v, "->", p)

FULL_TFL_READY = all(
    p is not None and Path(p).exists()
    for p in VARIANT_SOURCE_INPUTS.values()
)

print("\nFULL_TFL_READY =", FULL_TFL_READY)
if not FULL_TFL_READY:
    print(
        "Phase A is valid, but full record-level TFL execution is gated until all six "
        "original V0–V5 source inputs are resolved."
    )

V0 -> None
V1 -> /content/drive/MyDrive/MyDrive fhir_omop_colab /V1_missing_demographics_clinical_core_25k
V2 -> /content/drive/MyDrive/MyDrive fhir_omop_colab /V2_duplicate_encounter_ids_clinical_core_25k
V3 -> /content/drive/MyDrive/MyDrive fhir_omop_colab /V3_conflicting_codings_clinical_core_25k
V4 -> /content/drive/MyDrive/MyDrive fhir_omop_colab /V4_missing_medications_clinical_core_25k
V5 -> None

FULL_TFL_READY = False
Phase A is valid, but full record-level TFL execution is gated until all six original V0–V5 source inputs are resolved.


# Phase C

## Fidelity categories and warning codes

In [13]:
class Fate(str, Enum):
    PRESERVED = "PRESERVED"
    NORMALIZED = "NORMALIZED"
    COLLAPSED = "COLLAPSED"
    AMBIGUOUS = "AMBIGUOUS"
    UNMAPPED = "UNMAPPED"
    DROPPED = "DROPPED"
    TRACEABILITY_LOSS = "TRACEABILITY_LOSS"

WARNING_CODES = [
    "W_DEMOGRAPHIC_MISSING",
    "W_DUPLICATE_SOURCE_ID",
    "W_TRACEABILITY_LOSS",
    "W_CONFLICTING_CODING",
    "W_UNMAPPED_RESOURCE",
    "W_MEDICATION_ATTRIBUTION",
    "W_UNRESOLVED_REFERENCE",
]

FATE_PRIORITY = {
    Fate.PRESERVED.value: 0,
    Fate.NORMALIZED.value: 1,
    Fate.COLLAPSED.value: 2,
    Fate.DROPPED.value: 3,
    Fate.AMBIGUOUS.value: 4,
    Fate.UNMAPPED.value: 5,
    Fate.TRACEABILITY_LOSS.value: 6,
}

def choose_fate(*statuses):
    vals = []
    for s in statuses:
        if not s:
            continue
        vals.append(s.value if isinstance(s, Fate) else str(s))
    return max(vals, key=lambda x: FATE_PRIORITY.get(x, -1)) if vals else Fate.PRESERVED.value

TRANSFORMATION_VERSION = (
    f"TFL-0.1|pyomop-{getattr(pyomop,'__version__','unknown')}"
    f"|fhiry-{getattr(fhiry,'__version__','unknown')}"
)

print(TRANSFORMATION_VERSION)

TFL-0.1|pyomop-6.4.0|fhiry-5.2.2


## Mapping rules

In [14]:
PYOMOP_DIR = Path(pyomop.__file__).resolve().parent
MAPPING_PATH = PYOMOP_DIR / "mapping.default.json"

if not MAPPING_PATH.exists():
    raise FileNotFoundError(f"pyOMOP mapping.default.json not found: {MAPPING_PATH}")

with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    PYOMOP_MAPPING = json.load(f)

def infer_resource_type(mapping_table):
    for flt in mapping_table.get("filters", []) or []:
        if flt.get("column") in {"resourceType", "resource.resourceType"} and "equals" in flt:
            return str(flt["equals"])
    return None

MAPPING_RULES = []
for ordinal, table_map in enumerate(PYOMOP_MAPPING.get("tables", []), start=1):
    rtype = infer_resource_type(table_map)
    if not rtype:
        continue

    target = table_map["name"]
    MAPPING_RULES.append({
        "rule_order": ordinal,
        "source_resource_type": rtype,
        "target_omop_table": target,
        "mapping_rule_id": f"{rtype}_to_{target}_v1_r{ordinal:02d}",
        "columns": table_map.get("columns", {}),
        "filters": table_map.get("filters", []),
    })

mapping_rules_df = pd.DataFrame([
    {
        "rule_order": r["rule_order"],
        "source_resource_type": r["source_resource_type"],
        "target_omop_table": r["target_omop_table"],
        "mapping_rule_id": r["mapping_rule_id"],
        "source_paths_used": "|".join(sorted({
            v for v in r["columns"].values()
            if isinstance(v, str) and v
        })),
    }
    for r in MAPPING_RULES
])

display(mapping_rules_df)
mapping_rules_df.to_csv(PUBLIC_OUT / "tfl_mapping_rules.csv", index=False)

,rule_order,source_resource_type,target_omop_table,mapping_rule_id,source_paths_used
0,1,Patient,person,Patient_to_person_v1_r01,birthDate|extension|gender|patientId
1,2,Encounter,visit_occurrence,Encounter_to_visit_occurrence_v1_r02,class.code|hospitalization.dischargeDispositio...
2,3,Condition,condition_occurrence,Condition_to_condition_occurrence_v1_r03,abatementDateTime|clinicalStatus.coding.codes|...
3,4,Procedure,procedure_occurrence,Procedure_to_procedure_occurrence_v1_r04,code.coding.codes|patientId|performedPeriod.en...
4,5,Device,device_exposure,Device_to_device_exposure_v1_r05,distinctIdentifier|expirationDate|manufactureD...
5,6,Observation,measurement,Observation_to_measurement_v1_r06,code.coding.codes|effectiveDateTime|issued|pat...
6,7,Observation,observation,Observation_to_observation_v1_r07,code.coding.codes|effectiveDateTime|issued|pat...
7,8,AllergyIntolerance,observation,AllergyIntolerance_to_observation_v1_r08,code.coding.codes|patientId|reaction|recordedDate
8,9,Immunization,drug_exposure,Immunization_to_drug_exposure_v1_r09,occurrenceDateTime|patientId|vaccineCode.codin...
9,10,MedicationRequest,drug_exposure,MedicationRequest_to_drug_exposure_v1_r10,authoredOn|dosageInstruction|medicationCodeabl...


In [15]:
fate_table = pd.DataFrame([
    ["PRESERVED", "Source information remains represented without meaningful transformation-related loss."],
    ["NORMALIZED", "Source information is intentionally converted to a target-compatible representation."],
    ["COLLAPSED", "Multiple source representations are reduced to one retained target representation."],
    ["AMBIGUOUS", "Competing source representations exist and deterministic rules cannot treat the case as uniquely resolved."],
    ["UNMAPPED", "Source information has no implemented target mapping within the declared scope."],
    ["DROPPED", "Known source information is deliberately not retained by the current transformation path."],
    ["TRACEABILITY_LOSS", "The source object cannot be uniquely recovered through the retained source→target linkage."],
], columns=["fidelity_status","definition"])

warning_table = pd.DataFrame([
    ["W_DEMOGRAPHIC_MISSING", "Patient gender and/or birthDate missing at source.", "V1,V5"],
    ["W_DUPLICATE_SOURCE_ID", "Duplicate FHIR resource ID within resource type.", "V2"],
    ["W_TRACEABILITY_LOSS", "Missing/non-unique source ID or unresolved target link.", "V2"],
    ["W_CONFLICTING_CODING", "Condition contains multiple competing coding representations under the deterministic first-code path.", "V3,V5"],
    ["W_UNMAPPED_RESOURCE", "Resource type has no implemented active pyOMOP mapping rule.", "Any"],
    ["W_MEDICATION_ATTRIBUTION", "MedicationRequest lacks deterministic patient/code/date information used by the current mapping path.", "V4"],
    ["W_UNRESOLVED_REFERENCE", "FHIR reference points to a resource absent from the available export.", "External/Future"],
], columns=["warning_code","deterministic_trigger","expected_variant"])

display(fate_table)
display(warning_table)

fate_table.to_csv(PUBLIC_OUT / "tfl_information_fate_taxonomy.csv", index=False)
warning_table.to_csv(PUBLIC_OUT / "tfl_warning_rules.csv", index=False)

,fidelity_status,definition
0,PRESERVED,Source information remains represented without...
1,NORMALIZED,Source information is intentionally converted ...
2,COLLAPSED,Multiple source representations are reduced to...
3,AMBIGUOUS,Competing source representations exist and det...
4,UNMAPPED,Source information has no implemented target m...
5,DROPPED,Known source information is deliberately not r...
6,TRACEABILITY_LOSS,The source object cannot be uniquely recovered...


,warning_code,deterministic_trigger,expected_variant
0,W_DEMOGRAPHIC_MISSING,Patient gender and/or birthDate missing at sou...,"V1,V5"
1,W_DUPLICATE_SOURCE_ID,Duplicate FHIR resource ID within resource type.,V2
2,W_TRACEABILITY_LOSS,Missing/non-unique source ID or unresolved tar...,V2
3,W_CONFLICTING_CODING,Condition contains multiple competing coding r...,"V3,V5"
4,W_UNMAPPED_RESOURCE,Resource type has no implemented active pyOMOP...,Any
5,W_MEDICATION_ATTRIBUTION,MedicationRequest lacks deterministic patient/...,V4
6,W_UNRESOLVED_REFERENCE,FHIR reference points to a resource absent fro...,External/Future


# Phase D

## FHIR normalization

In [16]:
import fhiry.parallel as fp

FHIRY_CONFIG = {"REMOVE": ["text.div", "meta"], "RENAME": {}}

RESOURCE_TYPE_CANDIDATES = ["resourceType", "resource.resourceType"]
RESOURCE_ID_CANDIDATES = ["id", "resource.id", "resourceId", "resource.resourceId"]
PATIENT_ID_CANDIDATES = ["patientId", "patient_id", "resource.patientId"]

def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def load_source_dataframe(path):
    path = Path(path)

    if path.is_dir():
        if list(path.glob("*.ndjson")):
            df = fp.ndjson(str(path), config_json=json.dumps(FHIRY_CONFIG))
        else:
            df = fp.process(str(path), config_json=json.dumps(FHIRY_CONFIG))
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    elif path.suffix.lower() in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported source input: {path}")

    if df.empty:
        raise ValueError(f"No rows produced from {path}")

    rt_col = first_existing_column(df, RESOURCE_TYPE_CANDIDATES)
    id_col = first_existing_column(df, RESOURCE_ID_CANDIDATES)

    if rt_col is None:
        raise KeyError("FHIR resource type column not found.")

    df = df.copy()
    df["__tfl_resource_type"] = df[rt_col].astype(str)
    if id_col:
        df["__tfl_resource_id"] = df[id_col].astype("string")
    else:
        df["__tfl_resource_id"] = pd.Series([pd.NA] * len(df), dtype="string")

    df["__tfl_source_df_index"] = np.arange(len(df), dtype=np.int64)
    return df

def source_inventory(df):
    return (
        df.groupby("__tfl_resource_type", dropna=False)
        .size()
        .rename("source_rows")
        .reset_index()
        .rename(columns={"__tfl_resource_type":"resource_type"})
        .sort_values("source_rows", ascending=False)
    )

## Warning functions

In [17]:
def is_missing_value(v):
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except Exception:
        pass
    s = str(v).strip()
    return s == "" or s.lower() in {"nan", "none", "null", "<na>"}

def parse_multi_value(v):
    if is_missing_value(v):
        return []

    if isinstance(v, (list, tuple, set)):
        vals = list(v)
    else:
        s = str(v).strip()
        vals = None

        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    vals = list(parsed)
            except Exception:
                vals = None

        if vals is None:
            if "," in s:
                vals = [x.strip() for x in s.split(",")]
            elif "|" in s:
                vals = [x.strip() for x in s.split("|")]
            elif ";" in s:
                vals = [x.strip() for x in s.split(";")]
            else:
                vals = [s]

    out = []
    for x in vals:
        if not is_missing_value(x):
            out.append(str(x).strip().strip("'\""))
    return out

def unique_nonempty(values):
    return sorted(set(x for x in values if x and not is_missing_value(x)))

def row_first_value(row, candidates):
    for c in candidates:
        if c in row.index and not is_missing_value(row[c]):
            return row[c]
    return None

def coding_profile(row, resource_type):
    if resource_type == "Condition":
        code_cols = [
            "code.coding.codes", "code.coding.code",
            "resource.code.coding.codes", "resource.code.coding.code",
        ]
        system_cols = [
            "code.coding.systems", "code.coding.system",
            "resource.code.coding.systems", "resource.code.coding.system",
        ]
    elif resource_type == "MedicationRequest":
        code_cols = [
            "medicationCodeableConcept.coding.codes",
            "medicationCodeableConcept.coding.code",
            "resource.medicationCodeableConcept.coding.codes",
            "resource.medicationCodeableConcept.coding.code",
        ]
        system_cols = [
            "medicationCodeableConcept.coding.systems",
            "medicationCodeableConcept.coding.system",
            "resource.medicationCodeableConcept.coding.systems",
            "resource.medicationCodeableConcept.coding.system",
        ]
    else:
        code_cols = ["code.coding.codes", "code.coding.code", "resource.code.coding.code"]
        system_cols = ["code.coding.systems", "code.coding.system", "resource.code.coding.system"]

    codes, systems = [], []
    for c in code_cols:
        if c in row.index:
            codes.extend(parse_multi_value(row[c]))
    for c in system_cols:
        if c in row.index:
            systems.extend(parse_multi_value(row[c]))

    return unique_nonempty(codes), unique_nonempty(systems)

def duplicate_id_mask(df):
    valid = (
        df["__tfl_resource_id"].notna()
        & (df["__tfl_resource_id"].astype(str).str.strip() != "")
    )
    keys = df.loc[valid, ["__tfl_resource_type","__tfl_resource_id"]].astype(str)
    dup = keys.duplicated(keep=False)

    out = pd.Series(False, index=df.index)
    out.loc[keys.index] = dup.values
    return out

REFERENCE_RE = re.compile(r"([A-Za-z][A-Za-z0-9]+)/([^/\s?#]+)")

def build_resource_index(df):
    idx = set()
    for rt, rid in zip(df["__tfl_resource_type"], df["__tfl_resource_id"]):
        if not is_missing_value(rt) and not is_missing_value(rid):
            idx.add((str(rt), str(rid)))
    return idx

def unresolved_reference_count(row, resource_index):
    n = 0
    for col in row.index:
        if "reference" not in str(col).lower():
            continue
        for val in parse_multi_value(row[col]):
            match = REFERENCE_RE.search(str(val))
            if match and (match.group(1), match.group(2)) not in resource_index:
                n += 1
    return n

## Audit-event construction

In [18]:
def apply_simple_filters(df, filters):
    if not filters:
        return df

    mask = pd.Series(True, index=df.index)

    for flt in filters:
        col = flt.get("column")
        if not col:
            continue

        actual = col
        if actual not in df.columns and col == "resourceType":
            actual = "__tfl_resource_type"

        if actual not in df.columns:
            continue

        if "equals" in flt:
            mask &= df[actual].astype(str) == str(flt["equals"])
        elif flt.get("not_empty"):
            mask &= df[actual].notna() & (df[actual].astype(str).str.strip() != "")

    return df.loc[mask].copy()

def mapped_source_paths(rule):
    return sorted({
        v for v in rule["columns"].values()
        if isinstance(v, str) and v
    })

def deterministic_transformation_id(variant, source_type, source_id, rule_id, source_df_index):
    name = f"{variant}|{source_type}|{source_id}|{rule_id}|{source_df_index}"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, name))

def add_warning(existing, new_warning):
    current = set(x for x in str(existing or "").split("|") if x and x != "nan")
    current.add(new_warning)
    return "|".join(sorted(current))

def build_preload_audit(df, variant, run_reference_checks=True):
    resource_index = build_resource_index(df)
    dup_mask = duplicate_id_mask(df)
    mapped_resource_types = {r["source_resource_type"] for r in MAPPING_RULES}
    audit_rows = []

    for rule in MAPPING_RULES:
        df_rule = apply_simple_filters(df, rule["filters"])
        source_path_text = "|".join(mapped_source_paths(rule))

        for idx, row in df_rule.iterrows():
            rtype = str(row["__tfl_resource_type"])
            rid = row["__tfl_resource_id"]
            rid_text = None if is_missing_value(rid) else str(rid)
            codes, systems = coding_profile(row, rtype)

            warnings = []
            statuses = [Fate.NORMALIZED.value]

            if rid_text is None:
                warnings.append("W_TRACEABILITY_LOSS")
                statuses.append(Fate.TRACEABILITY_LOSS.value)

            if bool(dup_mask.loc[idx]):
                warnings.extend(["W_DUPLICATE_SOURCE_ID", "W_TRACEABILITY_LOSS"])
                statuses.append(Fate.TRACEABILITY_LOSS.value)

            if rtype == "Patient":
                gender = row_first_value(row, ["gender", "resource.gender"])
                birth = row_first_value(row, ["birthDate", "resource.birthDate"])
                if is_missing_value(gender) or is_missing_value(birth):
                    warnings.append("W_DEMOGRAPHIC_MISSING")

            if rtype == "Condition":
                if len(codes) > 1:
                    statuses.append(Fate.COLLAPSED.value)
                if len(codes) > 1 or len(systems) > 1:
                    warnings.append("W_CONFLICTING_CODING")
                    statuses.append(Fate.AMBIGUOUS.value)

            if rtype == "MedicationRequest":
                patient = row_first_value(
                    row,
                    PATIENT_ID_CANDIDATES + ["subject.reference", "resource.subject.reference"]
                )
                med_code = row_first_value(row, [
                    "medicationCodeableConcept.coding.codes",
                    "medicationCodeableConcept.coding.code",
                    "resource.medicationCodeableConcept.coding.codes",
                    "resource.medicationCodeableConcept.coding.code",
                ])
                authored = row_first_value(row, ["authoredOn", "resource.authoredOn"])

                if is_missing_value(patient) or is_missing_value(med_code) or is_missing_value(authored):
                    warnings.append("W_MEDICATION_ATTRIBUTION")

            unresolved_n = 0
            if run_reference_checks:
                unresolved_n = unresolved_reference_count(row, resource_index)
                if unresolved_n > 0:
                    warnings.append("W_UNRESOLVED_REFERENCE")

            source_df_index = int(row["__tfl_source_df_index"])

            audit_rows.append({
                "transformation_id": deterministic_transformation_id(
                    variant, rtype, rid_text, rule["mapping_rule_id"], source_df_index
                ),
                "variant": variant,
                "source_resource_type": rtype,
                "source_resource_id": rid_text,
                "source_reference_or_path": source_path_text,
                "source_value_or_code": None,
                "target_omop_table": rule["target_omop_table"],
                "target_omop_record_id": pd.NA,
                "target_omop_field": None,
                "mapping_rule_id": rule["mapping_rule_id"],
                "fidelity_status": choose_fate(*statuses),
                "warning_code": "|".join(sorted(set(warnings))),
                "transformation_version": TRANSFORMATION_VERSION,
                "source_df_index": source_df_index,
                "source_code_count": len(codes),
                "source_code_system_count": len(systems),
                "unresolved_reference_count": unresolved_n,
                "duplicate_source_id": bool(dup_mask.loc[idx]),
                "is_eligible_mapping": True,
                "target_link_method": None,
            })

    unsupported = df[~df["__tfl_resource_type"].isin(mapped_resource_types)]

    for idx, row in unsupported.iterrows():
        rtype = str(row["__tfl_resource_type"])
        rid = row["__tfl_resource_id"]
        rid_text = None if is_missing_value(rid) else str(rid)
        source_df_index = int(row["__tfl_source_df_index"])

        audit_rows.append({
            "transformation_id": deterministic_transformation_id(
                variant, rtype, rid_text, "UNMAPPED_RESOURCE", source_df_index
            ),
            "variant": variant,
            "source_resource_type": rtype,
            "source_resource_id": rid_text,
            "source_reference_or_path": None,
            "source_value_or_code": None,
            "target_omop_table": None,
            "target_omop_record_id": pd.NA,
            "target_omop_field": None,
            "mapping_rule_id": "UNMAPPED_RESOURCE",
            "fidelity_status": Fate.UNMAPPED.value,
            "warning_code": "W_UNMAPPED_RESOURCE",
            "transformation_version": TRANSFORMATION_VERSION,
            "source_df_index": source_df_index,
            "source_code_count": 0,
            "source_code_system_count": 0,
            "unresolved_reference_count": 0,
            "duplicate_source_id": False,
            "is_eligible_mapping": False,
            "target_link_method": None,
        })

    return pd.DataFrame(audit_rows)

# Phase E

## Fresh OMOP transformation and target-record linking

In [19]:
from pyomop import CdmEngineFactory
from pyomop.cdm54 import Base
from pyomop.loader import CdmCsvLoader
from pyomop.vocabulary import CdmVocabulary

ATHENA_VOCAB_DIR = None

FRESH_DB = {
    v: TFL_DRIVE_DIR / f"{v}_tfl_fresh_omop.sqlite"
    for v in ["V0","V1","V2","V3","V4","V5"]
}

PRIVATE_AUDIT = {
    v: TFL_DRIVE_DIR / f"{v}_tfl_full_audit.parquet"
    for v in ["V0","V1","V2","V3","V4","V5"]
}

async def fresh_pyomop_load(df, db_path, vocab_dir=None):
    db_path = Path(db_path)

    if db_path.exists():
        db_path.unlink()

    cdm = CdmEngineFactory(db="sqlite", name=str(db_path))
    await cdm.init_models(Base.metadata)

    if vocab_dir:
        vocab_dir = Path(vocab_dir)
        if not vocab_dir.exists():
            raise FileNotFoundError(f"Athena vocabulary directory not found: {vocab_dir}")
        vocab = CdmVocabulary(cdm, version="cdm54")
        await vocab.create_vocab(str(vocab_dir))

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_csv = Path(tmp.name)

    try:
        df.to_csv(temp_csv, index=False)
        loader = CdmCsvLoader(cdm, version="cdm54")
        await loader.load(
            csv_path=str(temp_csv),
            mapping_path=str(MAPPING_PATH),
            chunk_size=500,
        )
    finally:
        if temp_csv.exists():
            temp_csv.unlink()
        await cdm.dispose()

    if not db_path.exists() or db_path.stat().st_size == 0:
        raise RuntimeError(f"Fresh pyOMOP DB not created correctly: {db_path}")

    return db_path

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio
            nest_asyncio.apply()
            return loop.run_until_complete(coro)
    except RuntimeError:
        pass
    return asyncio.run(coro)

In [20]:
def single_primary_key(conn, table):
    pks = primary_key_columns(conn, table)
    return pks[0] if len(pks) == 1 else None

def resolve_target_ids_by_fresh_insert_order(audit, db_path):
    out = audit.copy()
    out["target_omop_record_id"] = pd.Series([pd.NA] * len(out), dtype="Int64")
    out["target_link_method"] = None

    with sqlite3.connect(db_path) as conn:
        target_tables = sorted(set(out["target_omop_table"].dropna().astype(str)))

        for table in target_tables:
            event_mask = out["is_eligible_mapping"] & (out["target_omop_table"] == table)
            table_events = out.loc[event_mask].copy()

            if table_events.empty:
                continue

            if not table_exists(conn, table):
                for ix in table_events.index:
                    out.at[ix, "warning_code"] = add_warning(out.at[ix, "warning_code"], "W_TRACEABILITY_LOSS")
                    out.at[ix, "fidelity_status"] = Fate.TRACEABILITY_LOSS.value
                    out.at[ix, "target_link_method"] = "FAILED_TARGET_TABLE_MISSING"
                continue

            pk = single_primary_key(conn, table)
            if pk is None:
                for ix in table_events.index:
                    out.at[ix, "warning_code"] = add_warning(out.at[ix, "warning_code"], "W_TRACEABILITY_LOSS")
                    out.at[ix, "fidelity_status"] = Fate.TRACEABILITY_LOSS.value
                    out.at[ix, "target_link_method"] = "FAILED_PRIMARY_KEY_RESOLUTION"
                continue

            target_ids = [
                int(r[0]) for r in conn.execute(
                    f'SELECT "{pk}" FROM "{table}" ORDER BY "{pk}"'
                ).fetchall()
            ]

            rules_for_table = [
                r for r in MAPPING_RULES
                if r["target_omop_table"] == table
            ]

            expected_total = sum(
                int((table_events["mapping_rule_id"] == r["mapping_rule_id"]).sum())
                for r in rules_for_table
            )

            if expected_total != len(target_ids):
                for ix in table_events.index:
                    out.at[ix, "warning_code"] = add_warning(out.at[ix, "warning_code"], "W_TRACEABILITY_LOSS")
                    out.at[ix, "fidelity_status"] = Fate.TRACEABILITY_LOSS.value
                    out.at[ix, "target_link_method"] = "FAILED_ROW_COUNT_RECONCILIATION"
                continue

            cursor = 0

            for rule in rules_for_table:
                idxs = out.index[
                    out["is_eligible_mapping"]
                    & (out["target_omop_table"] == table)
                    & (out["mapping_rule_id"] == rule["mapping_rule_id"])
                ].tolist()

                n = len(idxs)
                ids = target_ids[cursor:cursor+n]
                cursor += n

                if len(ids) != n:
                    for ix in idxs:
                        out.at[ix, "warning_code"] = add_warning(out.at[ix, "warning_code"], "W_TRACEABILITY_LOSS")
                        out.at[ix, "fidelity_status"] = Fate.TRACEABILITY_LOSS.value
                        out.at[ix, "target_link_method"] = "FAILED_RULE_SLICE"
                    continue

                out.loc[idxs, "target_omop_record_id"] = pd.array(ids, dtype="Int64")
                out.loc[idxs, "target_link_method"] = "fresh_load_insert_order_verified"

    nonunique_or_missing = (
        out["is_eligible_mapping"]
        & (
            out["source_resource_id"].isna()
            | out["duplicate_source_id"].fillna(False)
            | out["target_omop_record_id"].isna()
        )
    )

    for ix in out.index[nonunique_or_missing]:
        out.at[ix, "warning_code"] = add_warning(out.at[ix, "warning_code"], "W_TRACEABILITY_LOSS")
        out.at[ix, "fidelity_status"] = choose_fate(
            out.at[ix, "fidelity_status"],
            Fate.TRACEABILITY_LOSS.value,
        )

    return out

## Run V0–V5

In [21]:
SOURCE_DF = {}
TFL_AUDIT = {}
FRESH_RECON_ROWS = []

if FULL_TFL_READY and ALL_VARIANTS_READY:
    for variant in ["V0","V1","V2","V3","V4","V5"]:
        print("\n" + "="*80)
        print("VARIANT", variant)

        df = load_source_dataframe(VARIANT_SOURCE_INPUTS[variant])
        SOURCE_DF[variant] = df

        print("Source rows:", len(df))
        display(source_inventory(df).head(30))

        preload = build_preload_audit(df, variant, run_reference_checks=True)

        print("Eligible mapping events:", int(preload["is_eligible_mapping"].sum()))
        print("Unsupported inventory rows:", int((~preload["is_eligible_mapping"]).sum()))

        fresh_db = run_async(
            fresh_pyomop_load(
                df,
                FRESH_DB[variant],
                vocab_dir=ATHENA_VOCAB_DIR,
            )
        )

        audit = resolve_target_ids_by_fresh_insert_order(preload, fresh_db)
        TFL_AUDIT[variant] = audit
        audit.to_parquet(PRIVATE_AUDIT[variant], index=False)

        with sqlite3.connect(fresh_db) as fresh_conn, sqlite3.connect(VALIDATED_DB[variant]) as valid_conn:
            for table in CORE_TABLES:
                fresh_n = (
                    scalar(fresh_conn, f'SELECT COUNT(*) FROM "{table}"')
                    if table_exists(fresh_conn, table) else None
                )
                valid_n = (
                    scalar(valid_conn, f'SELECT COUNT(*) FROM "{table}"')
                    if table_exists(valid_conn, table) else None
                )

                FRESH_RECON_ROWS.append({
                    "variant": variant,
                    "table": table,
                    "fresh_tfl_rows": fresh_n,
                    "validated_rows": valid_n,
                    "exact_match": fresh_n == valid_n,
                })

    fresh_reconciliation = pd.DataFrame(FRESH_RECON_ROWS)
    display(fresh_reconciliation)
    fresh_reconciliation.to_csv(
        PUBLIC_OUT / "tfl_fresh_vs_validated_row_reconciliation.csv",
        index=False,
    )
else:
    fresh_reconciliation = pd.DataFrame()
    print("Full TFL rerun skipped because FULL_TFL_READY=False.")
    print("Resolve VARIANT_SOURCE_INPUTS, then rerun from Phase B onward.")

Full TFL rerun skipped because FULL_TFL_READY=False.
Resolve VARIANT_SOURCE_INPUTS, then rerun from Phase B onward.


# Phase F

## Fidelity metrics

In [22]:
LOSS_STATUSES = {
    Fate.UNMAPPED.value,
    Fate.DROPPED.value,
    Fate.TRACEABILITY_LOSS.value,
}

def compute_fidelity_metrics(audit, variant):
    eligible = audit[audit["is_eligible_mapping"]].copy()

    if eligible.empty:
        return pd.DataFrame()

    eligible["has_warning"] = eligible["warning_code"].fillna("").astype(str).str.len() > 0

    eligible["lineage_valid"] = (
        eligible["source_resource_id"].notna()
        & ~eligible["duplicate_source_id"].fillna(False)
        & eligible["target_omop_record_id"].notna()
    )

    eligible["mapped_target"] = eligible["target_omop_record_id"].notna()
    eligible["is_loss"] = eligible["fidelity_status"].isin(LOSS_STATUSES)

    eligible["is_ambiguity_or_warning"] = (
        (eligible["fidelity_status"] == Fate.AMBIGUOUS.value)
        | eligible["has_warning"]
    )

    groups = [("OVERALL", eligible)]
    groups.extend((str(domain), sub) for domain, sub in eligible.groupby("target_omop_table"))

    rows = []
    for domain, sub in groups:
        rows.append({
            "variant": variant,
            "domain": domain,
            "eligible_transformed_items": len(sub),
            "lineage_coverage": sub["lineage_valid"].mean(),
            "source_mapping_coverage": sub["mapped_target"].mean(),
            "transformation_loss_rate": sub["is_loss"].mean(),
            "ambiguity_warning_rate": sub["is_ambiguity_or_warning"].mean(),
        })

    return pd.DataFrame(rows)

if TFL_AUDIT:
    fidelity_metrics = pd.concat(
        [compute_fidelity_metrics(a, v) for v, a in TFL_AUDIT.items()],
        ignore_index=True,
    )

    display(fidelity_metrics)
    fidelity_metrics.to_csv(
        PUBLIC_OUT / "tfl_fidelity_metrics_by_variant_domain.csv",
        index=False,
    )
else:
    fidelity_metrics = pd.DataFrame()
    print("No TFL audit available yet.")

No TFL audit available yet.


In [23]:
def explode_warnings(audit):
    rows = []
    for _, r in audit.iterrows():
        codes = [x for x in str(r.get("warning_code","")).split("|") if x and x != "nan"]
        for wc in codes:
            rows.append({
                "variant": r["variant"],
                "target_omop_table": r["target_omop_table"],
                "source_resource_type": r["source_resource_type"],
                "warning_code": wc,
            })
    return pd.DataFrame(rows)

if TFL_AUDIT:
    warning_long = pd.concat(
        [explode_warnings(a) for a in TFL_AUDIT.values()],
        ignore_index=True,
    )

    warning_counts = (
        warning_long.groupby(["variant","warning_code"], dropna=False)
        .size()
        .rename("warning_count")
        .reset_index()
    )

    warning_domain_counts = (
        warning_long.groupby(
            ["variant","target_omop_table","warning_code"],
            dropna=False,
        )
        .size()
        .rename("warning_count")
        .reset_index()
    )

    display(warning_counts)

    warning_counts.to_csv(
        PUBLIC_OUT / "tfl_warning_counts_by_variant.csv",
        index=False,
    )
    warning_domain_counts.to_csv(
        PUBLIC_OUT / "tfl_warning_counts_by_variant_domain.csv",
        index=False,
    )
else:
    warning_long = pd.DataFrame()
    warning_counts = pd.DataFrame()
    warning_domain_counts = pd.DataFrame()

## Variant warning comparison

In [24]:
EXPECTED_WARNINGS = {
    "V1": ["W_DEMOGRAPHIC_MISSING"],
    "V2": ["W_DUPLICATE_SOURCE_ID", "W_TRACEABILITY_LOSS"],
    "V3": ["W_CONFLICTING_CODING"],
    "V4": ["W_MEDICATION_ATTRIBUTION"],
    "V5": ["W_DEMOGRAPHIC_MISSING", "W_CONFLICTING_CODING"],
}

if len(warning_counts):
    lookup = defaultdict(int)
    for _, r in warning_counts.iterrows():
        lookup[(r["variant"], r["warning_code"])] = int(r["warning_count"])

    detection_rows = []

    for variant, expected in EXPECTED_WARNINGS.items():
        for warning in expected:
            v0_n = lookup[("V0", warning)]
            variant_n = lookup[(variant, warning)]

            detection_rows.append({
                "variant": variant,
                "expected_warning": warning,
                "v0_count": v0_n,
                "variant_count": variant_n,
                "increase_vs_v0": variant_n - v0_n,
                "detected_above_v0": variant_n > v0_n,
            })

    detection_summary = pd.DataFrame(detection_rows)
    display(detection_summary)

    warning_matrix = (
        warning_counts.pivot(
            index="variant",
            columns="warning_code",
            values="warning_count",
        )
        .fillna(0)
        .astype(int)
        .reindex(["V0","V1","V2","V3","V4","V5"])
    )

    display(warning_matrix)

    detection_summary.to_csv(
        PUBLIC_OUT / "tfl_expected_vs_observed_warning_detection.csv",
        index=False,
    )
    warning_matrix.to_csv(PUBLIC_OUT / "tfl_warning_matrix.csv")
else:
    detection_summary = pd.DataFrame()
    warning_matrix = pd.DataFrame()

# Phase G

## Analytical-stability summary

In [25]:
ANALYTICAL_DIR = (
    REPO_DIR
    / "results_reviewer_strengthening"
    / "fhir_omop_analytical_stability"
)

PHASE4_DIR = REPO_DIR / "results" / "phase4_analytical_stability"

analytical_files = sorted(
    list(ANALYTICAL_DIR.glob("*.csv"))
    + list(PHASE4_DIR.glob("*.csv"))
)

print("Committed analytical CSV files found:", len(analytical_files))
for p in analytical_files[:50]:
    print(" -", p.relative_to(REPO_DIR))

Committed analytical CSV files found: 22
 - results/phase4_analytical_stability/phase4_V2_encounter_identity_traceability.csv
 - results/phase4_analytical_stability/phase4_V3_condition_semantic_mapping_summary.csv
 - results/phase4_analytical_stability/phase4_V3_missing_V0_condition_source_values.csv
 - results/phase4_analytical_stability/phase4_V3_new_condition_source_values_vs_V0.csv
 - results/phase4_analytical_stability/phase4_V4_changed_common_drug_source_values_vs_V0.csv
 - results/phase4_analytical_stability/phase4_V4_drug_exposure_completeness_summary.csv
 - results/phase4_analytical_stability/phase4_V4_missing_V0_drug_source_values.csv
 - results/phase4_analytical_stability/phase4_V4_new_drug_source_values_vs_V0.csv
 - results/phase4_analytical_stability/phase4_analytical_divergence_table_for_heatmap.csv
 - results/phase4_analytical_stability/phase4_analytical_stability_metrics.csv
 - results/phase4_analytical_stability/phase4_demographic_drift_V1.csv
 - results/phase4_analyti

In [26]:
def get_person_ids(conn, table):
    if not table_exists(conn, table):
        return set()
    s = pd.read_sql_query(
        f'SELECT DISTINCT person_id FROM "{table}"',
        conn,
    )["person_id"]
    return set(s.dropna().astype(int).tolist())

def jaccard(a, b):
    a, b = set(a), set(b)
    return 1.0 if not (a | b) else len(a & b) / len(a | b)

with sqlite3.connect(VALIDATED_DB["V0"]) as conn0:
    v0_drug = get_person_ids(conn0, "drug_exposure")
    v0_condition = get_person_ids(conn0, "condition_occurrence")

level3_rows = []

for variant, path in VALIDATED_DB.items():
    with sqlite3.connect(path) as conn:
        persons = get_person_ids(conn, "person")
        drug = get_person_ids(conn, "drug_exposure")
        condition = get_person_ids(conn, "condition_occurrence")

        condition_unique = scalar(
            conn,
            """
            SELECT COUNT(
                DISTINCT COALESCE(
                    NULLIF(TRIM(CAST(condition_source_value AS TEXT)),''),
                    '__MISSING__'
                )
            )
            FROM condition_occurrence
            """
        )

        drug_unique = scalar(
            conn,
            """
            SELECT COUNT(
                DISTINCT COALESCE(
                    NULLIF(TRIM(CAST(drug_source_value AS TEXT)),''),
                    '__MISSING__'
                )
            )
            FROM drug_exposure
            """
        )

        level3_rows.append({
            "variant": variant,
            "persons": len(persons),
            "drug_exposed_persons": len(drug),
            "condition_persons": len(condition),
            "drug_jaccard_vs_v0": jaccard(v0_drug, drug),
            "condition_jaccard_vs_v0": jaccard(v0_condition, condition),
            "drug_prevalence_percent": 100.0 * len(drug) / len(persons) if persons else np.nan,
            "condition_unique_source_values": condition_unique,
            "drug_unique_source_values": drug_unique,
        })

level3_regression = pd.DataFrame(level3_rows)
display(level3_regression)

level3_regression.to_csv(
    PUBLIC_OUT / "tfl_level3_regression_bridge.csv",
    index=False,
)

,variant,persons,drug_exposed_persons,condition_persons,drug_jaccard_vs_v0,condition_jaccard_vs_v0,drug_prevalence_percent,condition_unique_source_values,drug_unique_source_values
0,V0,1071,609,769,1.0,1.0,56.862745,247,147
1,V1,1071,609,769,1.0,1.0,56.862745,247,147
2,V2,1071,609,769,1.0,1.0,56.862745,247,147
3,V4,1071,609,769,1.0,1.0,56.862745,247,147


## Distributional summary

In [27]:
def empirical_distribution(values):
    s = (
        pd.Series(values)
        .fillna("__MISSING__")
        .astype(str)
    )

    counts = s.value_counts()

    return (
        counts
        /
        counts.sum()
    )


def aligned_probabilities(
    a_values,
    b_values
):
    pa = empirical_distribution(
        a_values
    )

    pb = empirical_distribution(
        b_values
    )

    idx = sorted(
        set(pa.index)
        |
        set(pb.index)
    )

    return (
        pa.reindex(
            idx,
            fill_value=0
        ).values,
        pb.reindex(
            idx,
            fill_value=0
        ).values,
    )


def jsd(
    a_values,
    b_values
):
    if (
        len(a_values) == 0
        or
        len(b_values) == 0
    ):
        return np.nan

    p, q = aligned_probabilities(
        a_values,
        b_values
    )

    return float(
        jensenshannon(
            p,
            q,
            base=2
        ) ** 2
    )


def shannon_entropy(values):
    if len(values) == 0:
        return np.nan

    p = empirical_distribution(
        values
    ).values

    return float(
        -(
            p
            *
            np.log2(p)
        ).sum()
    )


def get_text_column(
    conn,
    table,
    column
):
    return (
        pd.read_sql_query(
            f'SELECT "{column}" FROM "{table}"',
            conn
        )[column]
        .fillna("__MISSING__")
        .astype(str)
        .tolist()
    )


def database_available(variant):
    path = VALIDATED_DB.get(variant)

    return bool(
        path
        and Path(path).exists()
        and Path(path).stat().st_size > 0
    )


distributional_rows = []
distributional_status_rows = []

if not database_available("V0"):
    raise FileNotFoundError(
        "V0 database is required for distributional comparisons."
    )


if database_available("V3"):
    with sqlite3.connect(
        VALIDATED_DB["V0"]
    ) as c0, sqlite3.connect(
        VALIDATED_DB["V3"]
    ) as c3:

        v0_condition_values = get_text_column(
            c0,
            "condition_occurrence",
            "condition_source_value"
        )

        v3_condition_values = get_text_column(
            c3,
            "condition_occurrence",
            "condition_source_value"
        )

    distributional_rows.append({
        "comparison": "V3 condition vs V0",
        "jsd": jsd(
            v0_condition_values,
            v3_condition_values
        ),
        "v0_entropy_bits": shannon_entropy(
            v0_condition_values
        ),
        "variant_entropy_bits": shannon_entropy(
            v3_condition_values
        ),
        "entropy_change_bits": (
            shannon_entropy(
                v3_condition_values
            )
            -
            shannon_entropy(
                v0_condition_values
            )
        ),
    })

    distributional_status_rows.append({
        "comparison": "V3 condition vs V0",
        "status": "COMPLETED",
    })

else:
    distributional_status_rows.append({
        "comparison": "V3 condition vs V0",
        "status": "SKIPPED_V3_DATABASE_NOT_RESOLVED",
    })


if database_available("V4"):
    with sqlite3.connect(
        VALIDATED_DB["V0"]
    ) as c0, sqlite3.connect(
        VALIDATED_DB["V4"]
    ) as c4:

        v0_drug_values = get_text_column(
            c0,
            "drug_exposure",
            "drug_source_value"
        )

        v4_drug_values = get_text_column(
            c4,
            "drug_exposure",
            "drug_source_value"
        )

    distributional_rows.append({
        "comparison": "V4 drug vs V0",
        "jsd": jsd(
            v0_drug_values,
            v4_drug_values
        ),
        "v0_entropy_bits": shannon_entropy(
            v0_drug_values
        ),
        "variant_entropy_bits": shannon_entropy(
            v4_drug_values
        ),
        "entropy_change_bits": (
            shannon_entropy(
                v4_drug_values
            )
            -
            shannon_entropy(
                v0_drug_values
            )
        ),
    })

    distributional_status_rows.append({
        "comparison": "V4 drug vs V0",
        "status": "COMPLETED",
    })

else:
    distributional_status_rows.append({
        "comparison": "V4 drug vs V0",
        "status": "SKIPPED_V4_DATABASE_NOT_RESOLVED",
    })


secondary_crosscheck = pd.DataFrame(
    distributional_rows
)

distributional_status = pd.DataFrame(
    distributional_status_rows
)

display(
    secondary_crosscheck
)

display(
    distributional_status
)

secondary_crosscheck.to_csv(
    PUBLIC_OUT
    / "tfl_secondary_distributional_crosscheck.csv",
    index=False
)

distributional_status.to_csv(
    PUBLIC_OUT
    / "tfl_distributional_status.csv",
    index=False
)

,comparison,jsd,v0_entropy_bits,variant_entropy_bits,entropy_change_bits
0,V4 drug vs V0,0.0,4.036536,4.036536,0.0


,comparison,status
0,V3 condition vs V0,SKIPPED_V3_DATABASE_NOT_RESOLVED
1,V4 drug vs V0,COMPLETED


# Phase H

## Figures

In [28]:
if len(fidelity_metrics):
    overall = (
        fidelity_metrics[fidelity_metrics["domain"]=="OVERALL"]
        .set_index("variant")
        .reindex(["V0","V1","V2","V3","V4","V5"])
    )

    ax = overall[
        [
            "lineage_coverage",
            "source_mapping_coverage",
            "transformation_loss_rate",
            "ambiguity_warning_rate",
        ]
    ].plot(kind="bar", figsize=(11,6))

    ax.set_ylabel("Rate")
    ax.set_xlabel("Variant")
    ax.set_ylim(0, 1.05)
    ax.set_title("Transformation Fidelity Layer — Primary Metrics by Variant")
    ax.legend(loc="best")

    plt.tight_layout()
    fig_path = FIG_DIR / "figure_tfl_primary_metrics_by_variant.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(fig_path)
else:
    print("Figure skipped: fidelity metrics not available yet.")

Figure skipped: fidelity metrics not available yet.


In [29]:
if len(warning_counts):
    wm = (
        warning_counts.pivot(
            index="variant",
            columns="warning_code",
            values="warning_count",
        )
        .fillna(0)
        .reindex(["V0","V1","V2","V3","V4","V5"])
    )

    ax = wm.plot(kind="bar", figsize=(12,6))
    ax.set_ylabel("Warning count")
    ax.set_xlabel("Variant")
    ax.set_title("TFL Rule-Based Warning Counts by Variant")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02,1))

    plt.tight_layout()
    fig_path = FIG_DIR / "figure_tfl_warning_counts_by_variant.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(fig_path)
else:
    print("Figure skipped: warning counts not available yet.")

Figure skipped: warning counts not available yet.


# Phase I

## Public audit sample

In [30]:
def public_safe_audit(audit, n=100, random_state=42):
    if audit.empty:
        return audit.copy()

    sample = audit.sample(
        min(n, len(audit)),
        random_state=random_state,
    ).copy()

    def hash_id(x):
        if is_missing_value(x):
            return None
        return hashlib.sha256(str(x).encode("utf-8")).hexdigest()[:16]

    sample["source_resource_id_hash"] = sample["source_resource_id"].apply(hash_id)

    drop_cols = [
        c for c in ["source_resource_id", "source_value_or_code"]
        if c in sample.columns
    ]
    return sample.drop(columns=drop_cols)

if TFL_AUDIT:
    public_samples = pd.concat(
        [
            public_safe_audit(audit, n=100).assign(variant=variant)
            for variant, audit in TFL_AUDIT.items()
        ],
        ignore_index=True,
    )

    public_samples.to_csv(
        PUBLIC_OUT / "tfl_public_safe_audit_sample.csv",
        index=False,
    )
    display(public_samples.head(20))
else:
    public_samples = pd.DataFrame()

## Environment manifest

In [31]:
environment_manifest = {
    "python": sys.version,
    "platform": platform.platform(),
    "pyomop": getattr(pyomop, "__version__", "unknown"),
    "fhiry": getattr(fhiry, "__version__", "unknown"),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "mapping_path": str(MAPPING_PATH),
    "mapping_sha256": sha256_file(MAPPING_PATH),
    "transformation_version": TRANSFORMATION_VERSION,
    "fhiry_config": FHIRY_CONFIG,
    "full_tfl_ready": FULL_TFL_READY,
}

with open(
    PUBLIC_OUT / "tfl_environment_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(environment_manifest, f, indent=2)

print(json.dumps(environment_manifest, indent=2))

{
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "pyomop": "6.4.0",
  "fhiry": "5.2.2",
  "pandas": "2.2.2",
  "numpy": "2.0.2",
  "mapping_path": "/usr/local/lib/python3.12/dist-packages/pyomop/mapping.default.json",
  "mapping_sha256": "a35ea4afa6a516c68441d704af5ccc6ed25871b9b20ded8463f6f5b8eb820456",
  "transformation_version": "TFL-0.1|pyomop-6.4.0|fhiry-5.2.2",
  "fhiry_config": {
    "REMOVE": [
      "text.div",
      "meta"
    ],
    "RENAME": {}
  },
  "full_tfl_ready": false
}


# Phase J

## Validation summary

In [32]:
gate_rows = []

v0_person_n = int(
    v0_table_counts.loc[
        v0_table_counts["table"]=="person",
        "rows",
    ].iloc[0]
)

gate_rows.append({
    "gate": "V0 sole baseline has 1,071 persons",
    "passed": v0_person_n == 1071,
    "detail": f"V0 person rows={v0_person_n}",
})

if TFL_AUDIT:
    recon_ok = bool(fresh_reconciliation["exact_match"].all())

    gate_rows.append({
        "gate": "Fresh TFL reruns reconcile with validated row counts",
        "passed": recon_ok,
        "detail": (
            f"{int(fresh_reconciliation['exact_match'].sum())}/"
            f"{len(fresh_reconciliation)} table checks exact"
        ),
    })

    overall = fidelity_metrics[fidelity_metrics["domain"]=="OVERALL"]
    min_mapping_coverage = overall["source_mapping_coverage"].min()

    gate_rows.append({
        "gate": "Eligible mapping events resolve to target records",
        "passed": bool(min_mapping_coverage == 1.0),
        "detail": f"minimum overall source_mapping_coverage={min_mapping_coverage:.6f}",
    })

    for variant, label in [
        ("V2", "V2 duplicate-ID / traceability warnings above V0"),
        ("V3", "V3 conflicting-coding warning above V0"),
        ("V4", "V4 medication-attribution warning above V0"),
        ("V5", "V5 demographic + coding warnings above V0"),
    ]:
        sub = detection_summary[detection_summary["variant"]==variant]
        passed = bool(len(sub) and sub["detected_above_v0"].all())

        detail = "; ".join(
            f"{r.expected_warning}:{r.variant_count} vs V0:{r.v0_count}"
            for r in sub.itertuples()
        )

        gate_rows.append({
            "gate": label,
            "passed": passed,
            "detail": detail,
        })
else:
    gate_rows.append({
        "gate": "Full TFL source-to-target validation",
        "passed": False,
        "detail": (
            "Not executed because all six original V0–V5 source inputs "
            "were not resolved."
        ),
    })

validation_summary = pd.DataFrame(gate_rows)
display(validation_summary)

validation_summary.to_csv(
    PUBLIC_OUT / "tfl_validation_summary.csv",
    index=False,
)

,gate,passed,detail
0,"V0 sole baseline has 1,071 persons",True,V0 person rows=1071
1,Full TFL source-to-target validation,False,Not executed because all six original V0–V5 so...


# Phase K

## Export package

In [33]:
PACKAGE_DIR = Path("/content/tfl_public_package")

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir(parents=True)

for p in PUBLIC_OUT.glob("*"):
    if p.is_file():
        shutil.copy2(p, PACKAGE_DIR / p.name)

for p in FIG_DIR.glob("figure_tfl_*.png"):
    if p.is_file():
        shutil.copy2(p, PACKAGE_DIR / p.name)

zip_path = shutil.make_archive(
    "/content/FHIRy_pyOMOP_TFL_public_outputs",
    "zip",
    PACKAGE_DIR,
)

print("Public result package:", zip_path)
print("GitHub-ready result folder:", PUBLIC_OUT)
print("Private full-audit folder:", TFL_DRIVE_DIR)

Public result package: /content/FHIRy_pyOMOP_TFL_public_outputs.zip
GitHub-ready result folder: /content/ohdsi-fhir-omop-showcase-demo/results/transformation_fidelity
Private full-audit folder: /content/drive/MyDrive/fhir_omop_colab/tfl_runs


## Repository files

Recommended repository structure:

```text
notebooks/
  FHIRy_pyOMOP_TFL_Colab_Clean.ipynb

results/
  transformation_fidelity/
    phaseA_v0_table_counts.csv
    phaseA_v0_demographic_sanity.csv
    phaseA_v0_gender_counts.csv
    phaseA_v0_mapping_sanity.csv
    phaseA_v0_primary_key_sanity.csv
    phaseA_v0_manifest.csv
    tfl_mapping_rules.csv
    tfl_information_fate_taxonomy.csv
    tfl_warning_rules.csv
    tfl_fidelity_metrics_by_variant_domain.csv
    tfl_warning_counts_by_variant.csv
    tfl_warning_counts_by_variant_domain.csv
    tfl_expected_vs_observed_warning_detection.csv
    tfl_fresh_vs_validated_row_reconciliation.csv
    tfl_level3_regression_bridge.csv
    tfl_secondary_distributional_crosscheck.csv
    tfl_distributional_status.csv
    tfl_validation_summary.csv
    tfl_environment_manifest.json
    tfl_public_safe_audit_sample.csv

figures/
  figure_tfl_primary_metrics_by_variant.png
  figure_tfl_warning_counts_by_variant.png
```

Keep raw FHIR files, full SQLite databases, vocabulary files, and full record-level audit files outside the public repository.